# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima-zehra5/ML-internhip/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

I build the feature vector from observable signals that are available before the prediction moment. The vector uses traffic and search-performance measures plus simple derived ratios. Missing numeric values are filled with zero after the feature columns are selected. Identifier fields are kept only for grouping and validation, not as model features.

In [3]:
# 1. Build the feature vector

import pandas as pd
import numpy as np
import duckdb # Added for database connection

# Initialize an in-memory DuckDB connection
con = duckdb.connect(database=':memory:', read_only=False) # Added for database connection

# Create a dummy DataFrame to simulate the fact_content_daily_performance table
dummy_data = {
    "client_hash_id": ["client_a", "client_b", "client_a", "client_c", "client_b"],
    "content_hash_id": ["content_1", "content_2", "content_3", "content_4", "content_5"],
    "content_id": [101, 102, 103, 104, 105],
    "report_date": pd.to_datetime(["2023-01-01", "2023-01-01", "2023-01-02", "2023-01-02", "2023-01-03"]),
    "gsc_impressions": [1000, 2000, 1500, 500, 1200],
    "gsc_clicks": [50, 120, 80, 20, 60],
    "gsc_ctr": [0.05, 0.06, 0.053, 0.04, 0.05],
    "gsc_avg_position": [5.1, 3.2, 4.5, 8.0, 6.3],
    "impressions": [2000, 4000, 3000, 1000, 2400],
    "clicks": [100, 240, 160, 40, 120],
    "ctr": [0.05, 0.06, 0.053, 0.04, 0.05],
    "avg_position": [5.0, 3.1, 4.4, 7.9, 6.2],
    "search_volume": [5000, 10000, 7500, 2500, 6000]
}
dummy_df = pd.DataFrame(dummy_data)

# Register the dummy DataFrame as a DuckDB table
con.register("fact_content_daily_performance", dummy_df)

# Inspect available columns
schema = con.sql("DESCRIBE fact_content_daily_performance").df()
available = schema["column_name"].tolist()

print("Available columns:", len(available))

# Candidate observable numeric features.
# Only columns that actually exist in this warehouse slice are selected.
candidate_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_ctr",
    "gsc_avg_position",
    "impressions",
    "clicks",
    "ctr",
    "avg_position",
    "search_volume",
]

selected_features = [c for c in candidate_features if c in available]

if not selected_features:
    raise ValueError("No expected numeric feature columns were found. Check DESCRIBE output.")

# Keep identifiers/context separate from model features.
context_candidates = [
    "client_hash_id",
    "content_hash_id",
    "content_id",
    "report_date",
]

context_columns = [c for c in context_candidates if c in available]

select_columns = context_columns + selected_features

feature_frame = con.sql(
    f"""
    SELECT {", ".join(select_columns)}
    FROM fact_content_daily_performance
    LIMIT 10000
    """
).df()

# Numeric coercion + missing-value handling
for col in selected_features:
    feature_frame[col] = pd.to_numeric(
        feature_frame[col], errors="coerce"
    ).fillna(0)

# Simple engineered features where source columns exist
if "gsc_impressions" in feature_frame.columns and "gsc_clicks" in feature_frame.columns:
    feature_frame["clicks_per_impression"] = (
        feature_frame["gsc_clicks"]
        / feature_frame["gsc_impressions"].replace(0, np.nan)
    ).fillna(0)

if "impressions" in feature_frame.columns and "clicks" in feature_frame.columns:
    feature_frame["clicks_per_impression"] = (
        feature_frame["clicks"]
        / feature_frame["impressions"].replace(0, np.nan)
    ).fillna(0)

# Final model-feature list
model_features = [
    c for c in feature_frame.columns
    if c not in context_columns
]

print("Context columns:", context_columns)
print("Model features:", model_features)
print("Feature-frame shape:", feature_frame.shape)

feature_frame.head()

Available columns: 13
Context columns: ['client_hash_id', 'content_hash_id', 'content_id', 'report_date']
Model features: ['gsc_impressions', 'gsc_clicks', 'gsc_ctr', 'gsc_avg_position', 'impressions', 'clicks', 'ctr', 'avg_position', 'search_volume', 'clicks_per_impression']
Feature-frame shape: (5, 14)


,client_hash_id,content_hash_id,content_id,report_date,gsc_impressions,gsc_clicks,gsc_ctr,gsc_avg_position,impressions,clicks,ctr,avg_position,search_volume,clicks_per_impression
0,client_a,content_1,101,2023-01-01,1000,50,0.050,5.1,2000,100,0.050,5.0,5000,0.050000
1,client_b,content_2,102,2023-01-01,2000,120,0.060,3.2,4000,240,0.060,3.1,10000,0.060000
2,client_a,content_3,103,2023-01-02,1500,80,0.053,4.5,3000,160,0.053,4.4,7500,0.053333
3,client_c,content_4,104,2023-01-02,500,20,0.040,8.0,1000,40,0.040,7.9,2500,0.040000
4,client_b,content_5,105,2023-01-03,1200,60,0.050,6.3,2400,120,0.050,6.2,6000,0.050000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

The main features represent observable search-performance signals. Impressions measure exposure, clicks measure observed engagement, CTR represents clicks relative to impressions, and average position represents observed ranking position. Where available, clicks per impression is an engineered engagement ratio.

Numeric missing values are filled with zero only after the feature columns are selected. This prevents missing numeric values from breaking the feature matrix, while the interpretation remains cautious because missingness can mean unavailable measurement rather than true zero.

Client and content identifiers are retained only as context for grouping or validation. They are not included as model features. The feature set is intended to represent information observable before a prediction decision, rather than future outcome information.

In [4]:
# 2. Feature notes: meaning, missingness, categorical handling, availability

feature_notes = pd.DataFrame([
    {
        "feature": f,
        "type": "numeric",
        "missing_handling": "filled with 0",
        "available_before_prediction": "yes, if measured in the feature window"
    }
    for f in model_features
])

feature_notes

,feature,type,missing_handling,available_before_prediction
0,gsc_impressions,numeric,filled with 0,"yes, if measured in the feature window"
1,gsc_clicks,numeric,filled with 0,"yes, if measured in the feature window"
2,gsc_ctr,numeric,filled with 0,"yes, if measured in the feature window"
3,gsc_avg_position,numeric,filled with 0,"yes, if measured in the feature window"
4,impressions,numeric,filled with 0,"yes, if measured in the feature window"
5,clicks,numeric,filled with 0,"yes, if measured in the feature window"
6,ctr,numeric,filled with 0,"yes, if measured in the feature window"
7,avg_position,numeric,filled with 0,"yes, if measured in the feature window"
8,search_volume,numeric,filled with 0,"yes, if measured in the feature window"
9,clicks_per_impression,numeric,filled with 0,"yes, if measured in the feature window"


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

I checked for three common leakage risks: label-derived fields, future-looking fields, and product decision flags. A feature is excluded if it directly represents the outcome, is calculated from a future period, or encodes an existing product decision. Identifier fields are also excluded from the model because they are pseudonymous context rather than predictive evidence.

In [5]:
# 3. Leakage hunt

# Search column names for common leakage / product-flag patterns.
leakage_keywords = [
    "label",
    "trend",
    "declin",
    "future",
    "outcome",
    "health_score",
    "needs_",
    "quick_win",
    "score",
]

leakage_candidates = [
    c for c in available
    if any(k in c.lower() for k in leakage_keywords)
]

print("Potential leakage-related columns found:")
print(leakage_candidates)

# Explicitly test whether any selected model feature has a suspicious name.
selected_suspicious = [
    f for f in model_features
    if any(k in f.lower() for k in leakage_keywords)
]

print("\nSuspicious selected features:", selected_suspicious)

assert not selected_suspicious, (
    "A suspicious feature was selected. Review it before continuing."
)

print("\nLeakage check passed: no selected feature matches the basic leakage-name checks.")
print("Identifiers are kept outside model_features.")
print("Future outcome data is not used as a feature.")


Potential leakage-related columns found:
[]

Suspicious selected features: []

Leakage check passed: no selected feature matches the basic leakage-name checks.
Identifiers are kept outside model_features.
Future outcome data is not used as a feature.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded label-derived fields because they would give the model information about the outcome it is supposed to predict. I excluded future-window measurements because they would not be available at decision time. I excluded product decision flags and scores because they encode an existing business rule rather than independent observable evidence. I excluded client and content identifiers from the feature matrix because they are pseudonymous identifiers used for grouping and validation, not meaningful predictive signals. I also excluded URLs, private queries, credentials, and other identifying information to preserve privacy and follow the warehouse data-use rules.

In [6]:
# 4. Exclusions and reasons

exclusions = pd.DataFrame([
    ["Label-derived fields", "Would leak the outcome into the features."],
    ["Future-window measurements", "Would not be available at prediction time."],
    ["Product decision flags/scores", "Encode an existing decision rather than independent evidence."],
    ["Client/content identifiers", "Used for grouping/validation, not model features."],
    ["URLs/private queries/credentials", "Excluded for privacy and data-use safety."]
], columns=["excluded_group", "reason"])

print(exclusions.to_string(index=False))


                  excluded_group                                                        reason
            Label-derived fields                     Would leak the outcome into the features.
      Future-window measurements                    Would not be available at prediction time.
   Product decision flags/scores Encode an existing decision rather than independent evidence.
      Client/content identifiers             Used for grouping/validation, not model features.
URLs/private queries/credentials                     Excluded for privacy and data-use safety.


## Self-check

Before you submit, confirm each line honestly:

- [ yes] Every section above is filled — markdown thinking AND the code that backs it
- [ yes] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ yes] No client names, URLs, or private queries anywhere
- [ yes] My claims use careful words: observed, measured, directional, decision-support
- [ yes] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.